In [ ]:
import mysql.connector
import pandas as pd

conexao = mysql.connector.connect(
    host="localhost",
    user="root",
    password="********",
    database="bobo_benz"
)
cursor = conexao.cursor()

df = pd.read_csv("../data/bobo_benz_workshop_data_limpo.csv")

df = df.astype(object).where(pd.notna(df), None)


In [2]:
df_clients = df[['Customer_ID', 'Customer_Name', 'Phone', 'Customer_Type', 'Whatsapp_Followup', 'Fleet_Name']].drop_duplicates(subset=['Customer_ID'])

for index, row in df_clients.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO Clients (Customer_ID, Customer_Name, Phone, Customer_Type, Whatsapp_Followup, Fleet_Name)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row['Customer_ID'], 
        row['Customer_Name'], 
        row['Phone'], 
        row['Customer_Type'], 
        row['Whatsapp_Followup'], 
        row['Fleet_Name']
    ))
print(f"{len(df_clients)} clientes inseridos/verificados.")

1000 clientes inseridos/verificados.


In [3]:
df_vehicle = df[['Customer_ID', 'Vehicle_Type', 'Plate_Number', 'Has_Insurance_Cover']].drop_duplicates(subset=['Plate_Number']).reset_index(drop=True)
df_vehicle['Vehicle_ID'] = df_vehicle.index + 1 

for i, row in df_vehicle.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO Vehicle (Vehicle_ID, Customer_ID, Vehicle_Type, Plate_Number, Has_Insurance_Cover)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        row['Vehicle_ID'], 
        row['Customer_ID'], 
        row['Vehicle_Type'], 
        row['Plate_Number'], 
        row['Has_Insurance_Cover']
    ))
print(f"{len(df_vehicle)} veículos inseridos/verificados.")

935 veículos inseridos/verificados.


In [4]:
df_Mechanic = df[['Mechanic_Name', 'Mechanic_Skill_Level']].drop_duplicates(subset=['Mechanic_Name']).reset_index(drop=True)
df_Mechanic['Mechanic_ID'] = df_Mechanic.index + 1

for i, row in df_Mechanic.iterrows():
    cursor.execute("""
        INSERT IGNORE INTO Mechanic (Mechanic_ID, Mechanic_Name, Skill_Level)
        VALUES (%s, %s, %s)
    """, (
        row['Mechanic_ID'], 
        row['Mechanic_Name'], 
        row['Mechanic_Skill_Level']
    ))
print(f"{len(df_Mechanic)} mecânicos inseridos/verificados.")

5 mecânicos inseridos/verificados.


In [5]:
df_materiais_unicos = df['Parts_Replaced'].dropna().unique()

for i, material in enumerate(df_materiais_unicos):
    cursor.execute("""
        INSERT IGNORE INTO Material (Material_ID, Material_Name)
        VALUES (%s, %s)
    """, (i + 1, material))
print(f"{len(df_materiais_unicos)} materiais inseridos/verificados.")


5 materiais inseridos/verificados.


In [6]:
df['Service_ID'] = df.index + 1

for index, row in df.iterrows():
    cursor.execute("SELECT Vehicle_ID FROM Vehicle WHERE Plate_Number = %s", (row['Plate_Number'],))
    res_v = cursor.fetchone()
    v_id = res_v[0] if res_v else None
    
    cursor.execute("SELECT Mechanic_ID FROM Mechanic WHERE Mechanic_Name = %s", (row['Mechanic_Name'],))
    res_m = cursor.fetchone()
    m_id = res_m[0] if res_m else None
    
    cursor.execute("""
        INSERT IGNORE INTO Services (
            Service_ID, Customer_ID, Vehicle_ID, Mechanic_ID, Material_ID,
            Service_Type, Complaint, Service_Date, Follow_Up_Date, Mechanic_Notes, 
            Customer_Notes, Home_Service, Towing_Required, Service_Package, Payment_Mode, 
            Status, Service_Time_Hours, Wait_Time_Mins, Service_Cost, Parts_Cost, 
            Labour_Cost, Discount_Given, Rating, Referral_Source, Workshop_Location, 
            Parts_Source, Spare_Parts_Availability, Request_For_Pickup, Service_Priority,
            Return_Visit, Remarks, Promo_Code_Used
        ) VALUES (
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, 
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s
        )
    """, (
        row['Service_ID'], 
        row['Customer_ID'], 
        v_id, 
        m_id, 
        None, 
        row['Service_Type'], 
        row['Complaint'], 
        row['Date'], 
        row['Follow_Up_Date'], 
        row['Mechanic_Notes'], 
        row['Customer_Notes'], 
        row['Home_Service'], 
        row['Towing_Required'], 
        row['Service_Package'], 
        row['Payment_Mode'], 
        row['Status'], 
        row['Service_Time_Hours'], 
        row['Wait_Time_Mins'], 
        row['Service_Cost'], 
        row['Parts_Cost'], 
        row['Labour_Cost'], 
        row['Discount_Given'], 
        row['Rating'], 
        row['Referral_Source'], 
        row['Workshop_Location'], 
        row['Parts_Source'], 
        row['Spare_Parts_Availability'], 
        row['Request_For_Pickup'], 
        row['Service_Priority'], 
        row['Return_Visit'], 
        row['Remarks'], 
        row['Promo_Code_Used']
    ))
print(f"{len(df)} serviços inseridos/verificados.")



1000 serviços inseridos/verificados.


In [7]:

df['Service_ID'] = df.index + 1

for index, row in df.iterrows():
    pecas_texto = row['Parts_Replaced']
    
    if pd.notna(pecas_texto) and str(pecas_texto).strip() != "":
        lista_pecas = str(pecas_texto).split(',')
        
        for peca_nome in lista_pecas:
            peca_nome = peca_nome.strip()
            cursor.execute("SELECT Material_ID FROM Material WHERE Material_Name = %s", (peca_nome,))
            res = cursor.fetchone()
            
            if res:
                material_id = res[0]
                cursor.execute("""
                    INSERT IGNORE INTO ServicesMaterial (Service_ID, Material_ID, Quantity_Used)
                    VALUES (%s, %s, %s)
                """, (row['Service_ID'], material_id, 1))

conexao.commit()
print("Tabela ServicesMaterial preenchida com sucesso!")

Tabela ServicesMaterial preenchida com sucesso!


In [8]:
conexao.commit()
print("Carga concluída com sucesso!")

cursor.close()
conexao.close()

Carga concluída com sucesso!
